In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection


In [ ]:
# cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection


In [3]:
import numpy as np
BASE_DIR = "./dataset_windows20/"

X_train = np.load(BASE_DIR + "train/X.npy")
y_train = np.load(BASE_DIR + "train/y_bin.npy")

X_val = np.load(BASE_DIR + "val/X.npy")
y_val = np.load(BASE_DIR + "val/y_bin.npy")

X_test = np.load(BASE_DIR + "test/X.npy")
y_test = np.load(BASE_DIR + "test/y_bin.npy")


In [4]:
X_train = X_train[:, :, 1:]
X_val   = X_val[:, :, 1:]
X_test  = X_test[:, :, 1:]

print(X_train.shape)


(279986, 20, 16)


In [5]:
X_train = X_train[:, :, 0:-1]
X_val   = X_val[:, :, 0:-1]
X_test  = X_test[:, :, 0:-1]
print(X_train.shape)

(279986, 20, 15)


In [ ]:
## Preprocessing

In [6]:
X_train_nom = X_train[y_train == 0]   # solo nominal

mean_feat = X_train_nom.mean(axis=(0,1))   # (F,)
std_feat  = X_train_nom.std(axis=(0,1)) + 1e-8


In [7]:
def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_train_s = scale_windows(X_train, mean_feat, std_feat)
X_val_s   = scale_windows(X_val,   mean_feat, std_feat)
X_test_s  = scale_windows(X_test,  mean_feat, std_feat)


In [8]:
import pywt
import numpy as np

def compute_dwt_windows(X, wavelet="db4", level=1):
    """
    X: (N_windows, window_size, n_channels) o (N_windows, window_size)
    return:
        A: Approximation coefficients (low-frequency)
        D: Detail coefficients (high-frequency, concatenated)
    """
    if X.ndim == 2:
        X = X[..., np.newaxis]

    N, W, F = X.shape

    A_list = []
    D_list = []

    for i in range(N):
        A_ch = []
        D_ch = []

        for ch in range(F):
            coeffs = pywt.wavedec(X[i, :, ch], wavelet, level=level)

            A = coeffs[0]                  # Approximation
            D = np.concatenate(coeffs[1:]) # All detail levels

            A_ch.append(A)
            D_ch.append(D)

        A_list.append(np.stack(A_ch, axis=1))
        D_list.append(np.stack(D_ch, axis=1))

    return np.array(A_list), np.array(D_list)
# A_windows, D_windows = compute_dwt_windows(X)
# print("A:", A_windows.shape)
# print("D:", D_windows.shape)
# import numpy as np

def extract_wavelet_features(D):
    """
    D: array shape (N, n_coeffs, n_channels)
    Returns: (N, 3*n_channels)
    """
    N, _, n_channels = D.shape
    features = []

    for ch in range(n_channels):
        D_ch = D[:, :, ch]

        std_feat = np.std(D_ch, axis=1)
        energy_feat = np.sum(D_ch**2, axis=1)
        max_feat = np.max(np.abs(D_ch), axis=1)

        features.append(std_feat)
        features.append(energy_feat)
        features.append(max_feat)

    return np.stack(features, axis=1)



In [9]:
A_train, D_train = compute_dwt_windows(X_train_s)
A_val,   D_val   = compute_dwt_windows(X_val_s)
A_test,  D_test  = compute_dwt_windows(X_test_s)


In [10]:
print(D_train.shape)
# (N_samples, n_coeffs, n_channels)


(279986, 13, 15)


In [11]:
D_train_feat = extract_wavelet_features(D_train)
D_val_feat = extract_wavelet_features(D_val)
D_test_feat = extract_wavelet_features(D_test)

In [12]:
print("Feature shape:", D_train_feat.shape)

Feature shape: (279986, 45)


In [13]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
D_train_fs = scaler.fit_transform(D_train_feat)
D_test_fs = scaler.transform(D_test_feat)
D_val_fs = scaler.transform(D_val_feat)


In [14]:
input_dim = D_train_fs.shape[1]
print(input_dim)

45


In [15]:
print("D_train_fs:", D_train_fs.shape)
print("D_val_fs:", D_val_fs.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


D_train_fs: (279986, 45)
D_val_fs: (59977, 45)
y_train: (279986,)
y_val: (59977,)


In [16]:
print(D_train_fs.dtype)
print(y_train.dtype)


float64
int64


In [ ]:
D_train_fs = D_train_fs.astype(np.float32)
D_val_fs   = D_val_fs.astype(np.float32)

y_train = y_train.astype(np.float32)
y_val   = y_val.astype(np.float32)


In [ ]:
D_train_fs = D_train_fs.astype(np.float32)
D_val_fs = D_val_fs.astype(np.float32)


In [ ]:
# mean = np.mean(D_train, axis=(0,1), keepdims=True)
# std  = np.std(D_train, axis=(0,1), keepdims=True) + 1e-8

# D_train_n = (D_train - mean) / std
# D_val_n   = (D_val   - mean) / std
# D_test_n  = (D_test  - mean) / std


In [17]:
import tensorflow as tf
from tensorflow.keras import layers, models
model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,017 (7.88 KB)

 Trainable params: 2,017 (7.88 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
history = model.fit(
    D_train_fs,
    y_train,
    validation_data=(D_val_fs, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

Epoch 1/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.8674 - loss: 0.3411 - val_accuracy: 0.9100 - val_loss: 0.2512
Epoch 2/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.9049 - loss: 0.2584 - val_accuracy: 0.9148 - val_loss: 0.2397
Epoch 3/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 24s 3ms/step - accuracy: 0.9082 - loss: 0.2522 - val_accuracy: 0.9182 - val_loss: 0.2364
Epoch 4/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.9104 - loss: 0.2468 - val_accuracy: 0.9211 - val_loss: 0.2328
Epoch 5/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.9129 - loss: 0.2423 - val_accuracy: 0.9220 - val_loss: 0.2287
Epoch 6/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.9135 - loss: 0.2412 - val_accuracy: 0.9217 - val_loss: 0.2327
Epoch 7/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - accuracy: 0.9156 - loss: 0.2369 - val_accuracy: 0.9243 - val_loss: 0.2240
Epoch 8/10
8750/8750 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step - accuracy: 0.9168 - loss: 0

In [19]:
import tensorflow as tf
print(tf.__version__)


2.19.0


In [26]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
loss, acc = model.evaluate(D_test_fs, y_test)
print(f"\nTest Accuracy: {acc:.4f}")

y_pred = (model.predict(D_test_fs) > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9308 - loss: 0.2160

Test Accuracy: 0.9261
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.99      0.93     30406
           1       0.99      0.86      0.92     29572

    accuracy                           0.93     59978
   macro avg       0.93      0.93      0.93     59978
weighted avg       0.93      0.93      0.93     59978


Confusion Matrix:
[[30139   267]
 [ 4165 25407]]


In [ ]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 13, 8)          │           368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 13, 8)          │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 13, 8)          │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 8)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,013 (7.87 KB)

 Trainable params: 665 (2.60 KB)

 Non-trainable params: 16 (64.00 B)

 Optimizer params: 1,332 (5.21 KB)

a idea es pasar de:

“usar todos los coeficientes wavelet”
a
“usar estadísticos físicos de los coeficientes wavelet”

Eso reduce dimensión, memoria y multiplicaciones drásticamente. Perfecto para STM32.

🔵 Opción 3 — Feature engineering ultra-ligero sobre D

Después de calcular la DWT y obtener los coeficientes D (detalle) por canal y por ventana:

En vez de alimentar todos los coeficientes, calculamos:

Para cada canal:

std(D) → mide dispersión (actividad vibratoria)

energy(D) → ∑ D² (energía de alta frecuencia)

max(|D|) → pico transitorio

🔎 ¿Por qué esto funciona?

Los coeficientes D capturan cambios rápidos.

Si hay evento/anomalía → sube la energía

Si hay vibración → sube la desviación

Si hay pico abrupto → sube el máximo absoluto

Estás convirtiendo la señal en descriptores físicos interpretables.

Eso es excelente para un paper y excelente para microcontrolador.